In [ ]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

In [ ]:
#hide
from fastbook import *

# Модель языка, разработанная с нуля.


Теперь мы готовы углубиться... глубоко в область глубокого обучения! Вы уже узнали, как обучать простую нейронную сеть, но как перейти от этого к созданию современных моделей? В этой части книги мы раскроем все секреты, начиная с языковых моделей.

В главе <<chapter_nlp>> вы увидели, как настроить предварительно обученную языковую модель для создания классификатора текста. В этой главе мы объясним вам, что именно находится внутри этой модели, и что такое рекуррентная нейронная сеть (RNN). Сначала давайте соберем данные, которые позволят нам быстро создать прототипы наших различных моделей.

## Данные


Когда мы начинаем работу над новой задачей, мы всегда сначала пытаемся найти самый простой набор данных, который позволит нам быстро и легко протестировать различные методы и интерпретировать результаты. Несколько лет назад, когда мы начали заниматься моделированием языка, мы не нашли ни одного набора данных, который бы позволил проводить быструю разработку прототипов, поэтому мы создали свой собственный. Мы назвали его "*Human Numbers*", и он просто содержит первые 10 000 чисел, записанных на английском языке.

j: Одна из самых распространенных практических ошибок, которую я часто наблюдаю даже среди опытных специалистов, – это неиспользование подходящих наборов данных в нужные моменты процесса анализа. В частности, большинство людей склонны начинать с наборов данных, которые слишком велики и слишком сложны.

Мы можем загрузить, извлечь и просмотреть наш набор данных, как обычно.

In [ ]:
from fastai.text.all import *
path = untar_data(URLs.HUMAN_NUMBERS)

In [ ]:
#hide
Path.BASE_PATH = path

In [ ]:
path.ls()

(#2) [Path('train.txt'),Path('valid.txt')]

Давайте откроем эти два файла и посмотрим, что в них содержится. Сначала мы объединим все тексты вместе и проигнорируем разделение на обучающую и проверочную выборки, которое задано в наборе данных (мы вернемся к этому позже):


In [ ]:
lines = L()
with open(path/'train.txt') as f: lines += L(*f.readlines())
with open(path/'valid.txt') as f: lines += L(*f.readlines())
lines

(#9998) ['one \n','two \n','three \n','four \n','five \n','six \n','seven \n','eight \n','nine \n','ten \n'...]

Мы берем все эти строки и объединяем их в один большой поток. Чтобы обозначить переход от одного числа к следующему, мы используем символ `.` в качестве разделителя:


In [ ]:
text = ' . '.join([l.strip() for l in lines])
text[:100]

'one . two . three . four . five . six . seven . eight . nine . ten . eleven . twelve . thirteen . fo'

Мы можем разделить этот набор данных на токены, используя пробелы в качестве разделителей:


In [ ]:
tokens = text.split(' ')
tokens[:10]

['one', '.', 'two', '.', 'three', '.', 'four', '.', 'five', '.']

Для преобразования в числовые значения, нам необходимо создать список всех уникальных токенов (наш *словарь*):


In [ ]:
vocab = L(*tokens).unique()
vocab

(#30) ['one','.','two','three','four','five','six','seven','eight','nine'...]

Затем мы можем преобразовать наши токены в числовые значения, используя словарь (vocab), где каждый токен будет сопоставлен с определенным индексом.

In [ ]:
word2idx = {w:i for i,w in enumerate(vocab)}
nums = L(word2idx[i] for i in tokens)
nums

(#63095) [0,1,2,1,3,1,4,1,5,1...]

Теперь, когда у нас есть небольшой набор данных, на котором построение модели языка должно быть относительно простой задачей, мы можем создать нашу первую модель.

## Наша первая языковая модель, разработанная с нуля.

Один из простых способов преобразовать это в нейронную сеть – это указать, что мы будем предсказывать каждое слово, основываясь на трех предыдущих словах. Мы можем создать список, состоящий из всех последовательностей из трех слов, которые будут использоваться в качестве независимых переменных, а следующее слово после каждой последовательности – в качестве зависимой переменной.

Мы можем сделать это с помощью обычного Python. Давайте сначала попробуем это с токенами, чтобы увидеть, как это будет выглядеть:


In [ ]:
L((tokens[i:i+3], tokens[i+3]) for i in range(0,len(tokens)-4,3))

(#21031) [(['one', '.', 'two'], '.'),(['.', 'three', '.'], 'four'),(['four', '.', 'five'], '.'),(['.', 'six', '.'], 'seven'),(['seven', '.', 'eight'], '.'),(['.', 'nine', '.'], 'ten'),(['ten', '.', 'eleven'], '.'),(['.', 'twelve', '.'], 'thirteen'),(['thirteen', '.', 'fourteen'], '.'),(['.', 'fifteen', '.'], 'sixteen')...]

Теперь мы будем работать с тензорами, содержащими числовые значения, которые, собственно, и будут использоваться моделью:

In [ ]:
seqs = L((tensor(nums[i:i+3]), nums[i+3]) for i in range(0,len(nums)-4,3))
seqs

(#21031) [(tensor([0, 1, 2]), 1),(tensor([1, 3, 1]), 4),(tensor([4, 1, 5]), 1),(tensor([1, 6, 1]), 7),(tensor([7, 1, 8]), 1),(tensor([1, 9, 1]), 10),(tensor([10,  1, 11]), 1),(tensor([ 1, 12,  1]), 13),(tensor([13,  1, 14]), 1),(tensor([ 1, 15,  1]), 16)...]

Мы можем легко обрабатывать данные пакетами, используя класс `DataLoader`. Для начала мы разделим последовательности случайным образом:


In [ ]:
bs = 64
cut = int(len(seqs) * 0.8)
dls = DataLoaders.from_dsets(seqs[:cut], seqs[cut:], bs=64, shuffle=False)

Мы можем создать архитектуру нейронной сети, которая принимает на вход три слова и выдает прогноз вероятности каждого возможного следующего слова из словаря. Мы будем использовать три стандартных линейных слоя, но с двумя изменениями.

Первое изменение заключается в том, что первый линейный слой будет использовать только векторное представление (embedding) первого слова в качестве активаций, второй слой будет использовать векторное представление второго слова плюс выходные активации первого слоя, а третий слой будет использовать векторное представление третьего слова плюс выходные активации второго слоя. Основной эффект этого заключается в том, что каждое слово интерпретируется в информационном контексте всех слов, предшествующих ему.

Второе изменение заключается в том, что каждый из этих трех слоев будет использовать одну и ту же матрицу весов. Влияние одного слова на активации, полученные от предыдущих слов, не должно меняться в зависимости от позиции этого слова. Другими словами, значения активаций будут меняться по мере прохождения данных через слои, но сами веса слоев не будут меняться от слоя к слою. Таким образом, один слой не обучается для конкретной позиции в последовательности; он должен уметь обрабатывать все позиции.

Поскольку веса слоев не меняются, можно рассматривать последовательные слои как "один и тот же слой", повторяющийся. Фактически, PyTorch позволяет это реализовать: мы можем создать один слой и использовать его несколько раз.

### Наша языковая модель, реализованная в PyTorch.


Теперь мы можем создать модуль языковой модели, о котором мы говорили ранее:


In [ ]:
class LMModel1(Module):
    def __init__(self, vocab_sz, n_hidden):
        self.i_h = nn.Embedding(vocab_sz, n_hidden)  
        self.h_h = nn.Linear(n_hidden, n_hidden)     
        self.h_o = nn.Linear(n_hidden,vocab_sz)
        
    def forward(self, x):
        h = F.relu(self.h_h(self.i_h(x[:,0])))
        h = h + self.i_h(x[:,1])
        h = F.relu(self.h_h(h))
        h = h + self.i_h(x[:,2])
        h = F.relu(self.h_h(h))
        return self.h_o(h)

Как вы видите, мы создали три слоя:

- Слой встраивания (`i_h`, для *входных* данных, поступающих на *скрытый* слой)
- Линейный слой для формирования активаций для следующего слова (`h_h`, для *скрытого* слоя, поступающего на *скрытый* слой)
- Заключительный линейный слой для предсказания четвертого слова (`h_o`, для *скрытого* слоя, поступающего на *выходной* слой)

Возможно, это будет легче представить в виде схемы, поэтому давайте определим простую схематическую модель базовых нейронных сетей.  Рисунок <<img_simple_nn>> показывает, как мы будем представлять нейронную сеть с одним скрытым слоем.

```
<img alt="Иллюстрация простой нейронной сети" width="400" src="images/att_00020.png" caption="Иллюстрация простой нейронной сети" id="img_simple_nn">
```

Каждая фигура представляет собой активации: прямоугольник для входных данных, круг для активаций скрытого (внутреннего) слоя и треугольник для активаций выходного слоя. Мы будем использовать эти фигуры (обобщенные в <<img_shapes>>) во всех схемах в этой главе.

<img alt="Фигуры, используемые в наших схематических изображениях" width="200" src="images/att_00021.png" id="img_shapes" caption="Фигуры, используемые в наших схематических изображениях">

Стрелка обозначает фактический процесс вычисления для каждого слоя, то есть линейный слой, за которым следует функция активации. Используя эту нотацию, <<lm_rep>> демонстрирует, как выглядит наша простая языковая модель.

<img alt="Изображение нашей базовой языковой модели" width="500" caption="Изображение нашей базовой языковой модели" id="lm_rep" src="images/att_00022.png">

Чтобы упростить понимание, мы убрали детали вычислений для каждого слоя из обозначений на стрелках. Мы также использовали цветовую кодировку для стрелок, чтобы все стрелки одного цвета соответствовали одной и той же матрице весов. Например, все входные слои используют одну и ту же матрицу внедрения, поэтому они все имеют один и тот же цвет (зеленый).

Давайте попробуем обучить эту модель и посмотрим, что из этого выйдет:

In [ ]:
learn = Learner(dls, LMModel1(len(vocab), 64), loss_func=F.cross_entropy, 
                metrics=accuracy)
learn.fit_one_cycle(4, 1e-3)

epoch,train_loss,valid_loss,accuracy,time
0,1.824297,1.970941,0.467554,00:02
1,1.386973,1.823242,0.467554,00:02
2,1.417556,1.654497,0.494414,00:02
3,1.376440,1.650849,0.494414,00:02


Чтобы понять, насколько хорошо работает эта модель, давайте посмотрим, что бы дала нам очень простая модель. В этом случае мы всегда можем предсказывать наиболее часто встречающийся токен, поэтому давайте выясним, какой токен чаще всего является целевым в нашем наборе данных для проверки:


In [ ]:
n,counts = 0,torch.zeros(len(vocab))
for x,y in dls.valid:
    n += y.shape[0]
    for i in range_of(vocab): counts[i] += (y==i).long().sum()
idx = torch.argmax(counts)
idx, vocab[idx.item()], counts[idx].item()/n

(tensor(29), 'thousand', 0.15165200855716662)

Наиболее часто встречающийся токен имеет индекс 29, что соответствует токену `thousand`. Если бы мы всегда предсказывали этот токен, точность была бы примерно 15%, но на самом деле результаты гораздо лучше!

A: Моя первая гипотеза заключалась в том, что разделитель будет самым распространенным токеном, поскольку для каждого числа существует свой разделитель. Но, посмотрев на список `tokens`, я вспомнил, что большие числа записываются словами, и, например, до 10 000 часто встречается слово "тысяча": "пять тысяч", "пять тысяч один", "пять тысяч два" и так далее. Ой! Анализ ваших данных – отличный способ заметить как тонкие нюансы, так и очевидные вещи.

Это неплохая отправная точка. Давайте посмотрим, как мы можем переписать этот код, используя цикл.

### Наша первая рекуррентная нейронная сеть.

Изучив код нашего модуля, мы можем упростить его, заменив дублирующийся код, который вызывает слои, циклом `for`. Помимо упрощения кода, это также даст нам возможность применять наш модуль к последовательностям токенов разной длины – мы не будем ограничены списками токенов длиной всего три элемента:

In [ ]:
class LMModel2(Module):
    def __init__(self, vocab_sz, n_hidden):
        self.i_h = nn.Embedding(vocab_sz, n_hidden)  
        self.h_h = nn.Linear(n_hidden, n_hidden)     
        self.h_o = nn.Linear(n_hidden,vocab_sz)
        
    def forward(self, x):
        h = 0
        for i in range(3):
            h = h + self.i_h(x[:,i])
            h = F.relu(self.h_h(h))
        return self.h_o(h)

Давайте убедимся, что мы получаем одинаковые результаты, используя эту рефакторизацию:


In [ ]:
learn = Learner(dls, LMModel2(len(vocab), 64), loss_func=F.cross_entropy, 
                metrics=accuracy)
learn.fit_one_cycle(4, 1e-3)

epoch,train_loss,valid_loss,accuracy,time
0,1.816274,1.964143,0.460185,00:02
1,1.423805,1.739964,0.473259,00:02
2,1.430327,1.685172,0.485382,00:02
3,1.388390,1.657033,0.470406,00:02


Мы также можем переработать наше графическое представление точно таким же образом, как показано в разделе <<basic_rnn>> (здесь мы также убираем детали, касающиеся размеров активаций, и используем те же цвета стрелок, что и в разделе <<lm_rep>>).

```markdown
<img alt="Базовая рекуррентная нейронная сеть" width="400" caption="Базовая рекуррентная нейронная сеть" id="basic_rnn" src="images/att_00070.png">
```

Вы увидите, что существует набор активаций, которые обновляются каждый раз в процессе выполнения цикла и хранятся в переменной `h` — это называется *скрытым состоянием*.


```markdown
> Жargon: скрытое состояние: Активации, которые обновляются на каждом шаге работы рекуррентной нейронной сети.
```

Нейронная сеть, определенная с использованием цикла, подобного этому, называется *рекуррентной нейронной сетью* (RNN). Важно понимать, что RNN – это не сложная новая архитектура, а просто переработка многослойной нейронной сети с использованием цикла `for`.

> A: Мое личное мнение: если бы их называли "нейронными сетями с циклами" или LNN, они казались бы на 50% менее сложными!

Теперь, когда мы знаем, что такое рекуррентная нейронная сеть (RNN), давайте попробуем сделать ее немного лучше.

## Улучшение рекуррентных нейронных сетей (RNN)


Изучая код нашей рекуррентной нейронной сети (RNN), можно заметить, что мы инициализируем скрытое состояние нулями для каждой новой входной последовательности. Почему это может быть проблемой? Мы сделали наши примеры последовательностей короткими, чтобы они легко помещались в пакеты. Но если правильно упорядочить примеры, модель будет считывать эти последовательности последовательно, что подвергнет модель воздействию длинных участков исходной последовательности.

Еще один аспект, который стоит рассмотреть, – это возможность получения большего объема информации: зачем предсказывать только четвертое слово, когда мы можем использовать промежуточные прогнозы для предсказания второго и третьего слов?

Давайте посмотрим, как мы можем реализовать эти изменения, начиная с добавления некоторой информации о состоянии.

### Поддержание состояния рекуррентной нейронной сети (RNN)


Поскольку мы инициализируем скрытое состояние модели нулями для каждого нового образца, мы теряем всю информацию о предложениях, которые мы видели до этого, что означает, что наша модель на самом деле не знает, на каком этапе мы находимся в общей последовательности подсчета. Это легко исправить; мы можем просто переместить инициализацию скрытого состояния в метод `__init__`.

Однако это исправление создаст свою собственную, хоть и тонкую, но важную проблему. Фактически, это сделает нашу нейронную сеть такой же глубокой, как общее количество токенов в нашем документе. Например, если в нашем наборе данных было 10 000 токенов, мы создадим нейронную сеть с 10 000 слоями.

Чтобы понять, почему это происходит, рассмотрим исходное графическое представление нашей рекуррентной нейронной сети (см. <<lm_rep>>), до того, как мы ее рефакторизовали с использованием цикла `for`. Вы можете видеть, что каждый слой соответствует одному входному токену. Когда мы говорим о представлении рекуррентной нейронной сети до ее рефакторинга с использованием цикла `for`, мы называем это *развернутым представлением*. Часто полезно рассматривать развернутое представление, когда вы пытаетесь понять работу RNN.

Проблема нейронной сети с 10 000 слоями заключается в том, что когда вы дойдете до 10 000-го слова в наборе данных, вам все равно придется вычислять производные, начиная с первого слоя. Это будет очень медленно и потребует большого объема памяти. Маловероятно, что вы сможете даже сохранить один мини-пакет в своей GPU.

Решение этой проблемы – сообщить PyTorch, что мы не хотим распространять производные через всю неявную нейронную сеть. Вместо этого мы будем сохранять только последние три слоя градиентов. Чтобы удалить всю историю градиентов в PyTorch, мы используем метод `detach`.

Вот новая версия нашей RNN. Теперь она является stateful, поскольку она запоминает свои активации между разными вызовами метода `forward`, которые представляют ее использование для разных образцов в пакете:


In [ ]:
class LMModel3(Module):
    def __init__(self, vocab_sz, n_hidden):
        self.i_h = nn.Embedding(vocab_sz, n_hidden)  
        self.h_h = nn.Linear(n_hidden, n_hidden)     
        self.h_o = nn.Linear(n_hidden,vocab_sz)
        self.h = 0
        
    def forward(self, x):
        for i in range(3):
            self.h = self.h + self.i_h(x[:,i])
            self.h = F.relu(self.h_h(self.h))
        out = self.h_o(self.h)
        self.h = self.h.detach()
        return out
    
    def reset(self): self.h = 0

Эта модель будет иметь одинаковые выходные значения независимо от выбранной длины последовательности, поскольку скрытое состояние будет запоминать последнее выходное значение из предыдущей порции данных. Единственное, что будет отличаться, это градиенты, вычисляемые на каждом шаге: они будут рассчитываться только для токенов, соответствующих длине последовательности, а не для всей последовательности целиком. Этот подход называется *обратное распространение ошибки во времени* (BPTT).

```
жаргон: Обратное распространение ошибки во времени (BPTT): рассматривается нейронная сеть, которая, по сути, имеет один слой для каждого временного шага (обычно реализованная с использованием цикла), как единая большая модель, и градиенты вычисляются для нее обычным способом. Чтобы избежать нехватки памяти и времени, мы обычно используем усеченное BPTT, которое "отделяет" историю вычислений в скрытом состоянии через определенные временные интервалы.
```

Чтобы использовать `LMModel3`, необходимо убедиться, что данные будут представлены в определенном порядке. Как мы видели в главе <<chapter_nlp>>, если первая строка первого пакета данных — это `dset[0]`, то первая строка второго пакета данных должна быть `dset[1]`, чтобы модель воспринимала текст как непрерывный поток.

В главе <<chapter_nlp>> `LMDataLoader` выполнял эту задачу за нас. В этот раз мы сделаем это сами.

Для этого мы переупорядочим наш набор данных. Сначала мы делим выборки на `m = len(dset) // bs` групп (это эквивалентно разделению всего объединенного набора данных, например, на 64 равноценные части, поскольку мы используем `bs=64`). `m` — это длина каждой из этих частей. Например, если мы используем весь наш набор данных (хотя мы вскоре разделим его на обучающую и проверочную выборки), то это будет:


In [ ]:
m = len(seqs)//bs
m,bs,len(seqs)

(328, 64, 21031)

Первая группа образцов будет состоять из элементов:

    (0, m, 2*m, ..., (bs-1)*m)

Вторая группа образцов:

    (1, m+1, 2*m+1, ..., (bs-1)*m+1)

И так далее. Таким образом, на каждой эпохе модель будет видеть непрерывный фрагмент текста размером `3*m` (поскольку каждый фрагмент текста имеет размер 3) на каждой строке в пакете данных.

Следующая функция выполняет переиндексацию:


In [ ]:
def group_chunks(ds, bs):
    m = len(ds) // bs
    new_ds = L()
    for i in range(m): new_ds += L(ds[i + m*j] for j in range(bs))
    return new_ds

Затем, при создании объектов `DataLoaders`, мы передаем параметр `drop_last=True`, чтобы отбрасывать последнюю партию данных, размер которой не соответствует заданному размеру пакета (`bs`). Мы также передаем параметр `shuffle=False`, чтобы обеспечить чтение текстов в определенном порядке:

In [ ]:
cut = int(len(seqs) * 0.8)
dls = DataLoaders.from_dsets(
    group_chunks(seqs[:cut], bs), 
    group_chunks(seqs[cut:], bs), 
    bs=bs, drop_last=True, shuffle=False)

Последнее, что мы добавляем, – небольшая модификация цикла обучения с помощью объекта `Callback`. Мы подробнее поговорим о коллбэках в главе <<chapter_accel_sgd>>; в данном случае этот коллбэк будет вызывать метод `reset` нашей модели в начале каждой эпохи и перед каждым этапом валидации. Поскольку мы реализовали этот метод для обнуления скрытого состояния модели, это гарантирует, что мы начинаем с чистого состояния перед обработкой этих непрерывных фрагментов текста. Мы также можем увеличить продолжительность обучения:


In [ ]:
learn = Learner(dls, LMModel3(len(vocab), 64), loss_func=F.cross_entropy,
                metrics=accuracy, cbs=ModelResetter)
learn.fit_one_cycle(10, 3e-3)

epoch,train_loss,valid_loss,accuracy,time
0,1.677074,1.827367,0.467548,00:02
1,1.282722,1.870913,0.388942,00:02
2,1.090705,1.651793,0.462500,00:02
3,1.005092,1.613794,0.516587,00:02
4,0.965975,1.560775,0.551202,00:02
5,0.916182,1.595857,0.560577,00:02
6,0.897657,1.539733,0.574279,00:02
7,0.836274,1.585141,0.583173,00:02
8,0.805877,1.629808,0.586779,00:02
9,0.795096,1.651267,0.588942,00:02


Это уже лучше! Следующий шаг – использовать больше целевых значений и сравнить их с промежуточными результатами.

### Увеличение уровня сигнала.


Еще одна проблема нашего текущего подхода заключается в том, что мы предсказываем только одно выходное слово для каждых трех входных слов. Это означает, что объем информации, который мы используем для обновления весов, не так велик, как мог бы быть. Было бы лучше предсказывать следующее слово после каждого слова, а не через каждые три слова, как показано в схеме <<stateful_rep>>.

```markdown
<img alt="RNN, предсказывающая после каждого токена" width="400" caption="RNN, предсказывающая после каждого токена" id="stateful_rep" src="images/att_00024.png">
```

Это достаточно просто добавить. Сначала нам нужно изменить наши данные таким образом, чтобы зависимая переменная содержала каждое из трех следующих слов после каждого из трех введенных слов. Вместо числа `3` мы используем атрибут `sl` (сокращение от "sequence length" - длина последовательности) и немного увеличиваем его:


In [ ]:
sl = 16
seqs = L((tensor(nums[i:i+sl]), tensor(nums[i+1:i+sl+1]))
         for i in range(0,len(nums)-sl-1,sl))
cut = int(len(seqs) * 0.8)
dls = DataLoaders.from_dsets(group_chunks(seqs[:cut], bs),
                             group_chunks(seqs[cut:], bs),
                             bs=bs, drop_last=True, shuffle=False)

Рассмотрев первый элемент массива `seqs`, мы видим, что он содержит два списка одинакового размера. Второй список идентичен первому, но сдвинут на один элемент:


In [ ]:
[L(vocab[o] for o in s) for s in seqs[0]]

[(#16) ['one','.','two','.','three','.','four','.','five','.'...],
 (#16) ['.','two','.','three','.','four','.','five','.','six'...]]

Теперь нам необходимо модифицировать нашу модель таким образом, чтобы она выдавала прогноз после каждого слова, а не только в конце последовательности из трех слов.

In [ ]:
class LMModel4(Module):
    def __init__(self, vocab_sz, n_hidden):
        self.i_h = nn.Embedding(vocab_sz, n_hidden)  
        self.h_h = nn.Linear(n_hidden, n_hidden)     
        self.h_o = nn.Linear(n_hidden,vocab_sz)
        self.h = 0
        
    def forward(self, x):
        outs = []
        for i in range(sl):
            self.h = self.h + self.i_h(x[:,i])
            self.h = F.relu(self.h_h(self.h))
            outs.append(self.h_o(self.h))
        self.h = self.h.detach()
        return torch.stack(outs, dim=1)
    
    def reset(self): self.h = 0

Эта модель будет возвращать результаты с формой `bs x sl x vocab_sz` (так как мы использовали объединение по размерности `dim=1`). Наши целевые значения имеют форму `bs x sl`, поэтому нам необходимо преобразовать их в одномерный вид перед использованием в функции `F.cross_entropy`:


In [ ]:
def loss_func(inp, targ):
    return F.cross_entropy(inp.view(-1, len(vocab)), targ.view(-1))

Теперь мы можем использовать эту функцию потерь для обучения модели:

In [ ]:
learn = Learner(dls, LMModel4(len(vocab), 64), loss_func=loss_func,
                metrics=accuracy, cbs=ModelResetter)
learn.fit_one_cycle(15, 3e-3)

epoch,train_loss,valid_loss,accuracy,time
0,3.103298,2.874341,0.212565,00:01
1,2.231964,1.971280,0.462158,00:01
2,1.711358,1.813547,0.461182,00:01
3,1.448516,1.828176,0.483236,00:01
4,1.288630,1.659564,0.520671,00:01
5,1.161470,1.714023,0.554932,00:01
6,1.055568,1.660916,0.575033,00:01
7,0.960765,1.719624,0.591064,00:01
8,0.870153,1.839560,0.614665,00:01
9,0.808545,1.770278,0.624349,00:01


Нам нужно тренироваться дольше, поскольку задача немного изменилась и стала сложнее. Но в итоге мы получаем хороший результат... По крайней мере, иногда. Если вы запустите программу несколько раз, вы увидите, что результаты могут сильно отличаться. Это связано с тем, что у нас, по сути, очень глубокая нейронная сеть, что может приводить к очень большим или очень маленьким градиентам. В следующей части этой главы мы рассмотрим, как с этим бороться.

Очевидный способ получить более качественную модель – это усложнить ее структуру, сделать ее "глубже": в нашей базовой рекуррентной нейронной сети (RNN) есть только один линейный слой между скрытым состоянием и выходными активациями, поэтому, возможно, мы получим лучшие результаты, если добавим больше таких слоев.

## Многослойные рекуррентные нейронные сети (RNN)


В многослойной рекуррентной нейронной сети (RNN) мы передаем выходные данные от одного рекуррентного нейронного слоя на вход другому, подобно тому, как это показано в примере <<stacked_rnn_rep>>.

```markdown
<img alt="2-слойная рекуррентная нейронная сеть" width="550" caption="2-слойная рекуррентная нейронная сеть" id="stacked_rnn_rep" src="images/att_00025.png">
```

Представление в развернутом виде показано на рисунке <<unrolled_stack_rep>> (аналогично рисунку <<lm_rep>>).

```markdown
<img alt="Двухслойная развернутая рекуррентная нейронная сеть" width="500" caption="Двухслойная развернутая рекуррентная нейронная сеть" id="unrolled_stack_rep" src="images/att_00026.png">
```

Давайте посмотрим, как это можно реализовать на практике.

### О модели


Мы можем сэкономить время, используя класс `RNN` библиотеки PyTorch, который реализует то, что мы создали ранее, но также предоставляет возможность объединять несколько экземпляров RNN, как мы обсуждали:


In [ ]:
class LMModel5(Module):
    def __init__(self, vocab_sz, n_hidden, n_layers):
        self.i_h = nn.Embedding(vocab_sz, n_hidden)
        self.rnn = nn.RNN(n_hidden, n_hidden, n_layers, batch_first=True)
        self.h_o = nn.Linear(n_hidden, vocab_sz)
        self.h = torch.zeros(n_layers, bs, n_hidden)
        
    def forward(self, x):
        res,h = self.rnn(self.i_h(x), self.h)
        self.h = h.detach()
        return self.h_o(res)
    
    def reset(self): self.h.zero_()

In [ ]:
learn = Learner(dls, LMModel5(len(vocab), 64, 2), 
                loss_func=CrossEntropyLossFlat(), 
                metrics=accuracy, cbs=ModelResetter)
learn.fit_one_cycle(15, 3e-3)

epoch,train_loss,valid_loss,accuracy,time
0,3.055853,2.591640,0.437907,00:01
1,2.162359,1.787310,0.471598,00:01
2,1.710663,1.941807,0.321777,00:01
3,1.520783,1.999726,0.312012,00:01
4,1.330846,2.012902,0.413249,00:01
5,1.163297,1.896192,0.450684,00:01
6,1.033813,2.005209,0.434814,00:01
7,0.919090,2.047083,0.456706,00:01
8,0.822939,2.068031,0.468831,00:01
9,0.750180,2.136064,0.475098,00:01


Вот это разочаровывает... наша предыдущая однослойная рекуррентная нейронная сеть (RNN) показала лучшие результаты. Почему? Причина в том, что у нас более глубокая модель, что приводит к взрыву или затуханию активаций.

### Внезапное прекращение или исчезновение активности.

На практике создание точных моделей с использованием рекуррентных нейронных сетей (RNN) представляет собой сложную задачу. Мы получим лучшие результаты, если будем реже вызывать функцию `detach` и использовать больше слоев – это позволит нашей RNN иметь больший временной горизонт для обучения и создавать более богатые признаки. Однако это также означает, что нам придется обучать более сложную модель. Ключевой проблемой в разработке глубокого обучения является поиск способов обучения таких моделей.

Сложность заключается в том, что происходит при многократном умножении на матрицу. Представьте, что происходит при многократном умножении на число. Например, если умножить на 2, начиная с 1, мы получим последовательность 1, 2, 4, 8 и т.д. Уже через 32 шага мы достигнем числа 4 294 967 296. Аналогичная проблема возникает при умножении на 0,5: мы получаем 0,5, 0,25, 0,125 и т.д., и через 32 шага это число станет 0,00000000023. Как видно, даже небольшое отклонение от единицы при умножении приводит к экспоненциальному росту или исчезновению исходного числа после нескольких повторений.

Поскольку умножение матриц — это просто умножение чисел и их суммирование, то же самое происходит при многократном умножении матриц. Глубокая нейронная сеть – это, по сути, последовательность таких умножений, где каждый дополнительный слой представляет собой еще одно умножение матриц. Это означает, что глубокая нейронная сеть очень легко может получить чрезвычайно большие или чрезвычайно малые числа.

Это проблема, потому что способ хранения чисел в компьютерах (известный как "числа с плавающей запятой") приводит к тому, что точность этих чисел уменьшается по мере удаления от нуля. Диаграмма, представленная в разделе <<float_prec>>, из отличной статьи ["What You Never Wanted to Know About Floating Point but Will Be Forced to Find Out"](http://www.volkerschatz.com/science/float.html), показывает, как точность чисел с плавающей запятой меняется на числовой прямой.

```markdown
<img alt="Точность чисел с плавающей точкой" width="1000" caption="Точность чисел с плавающей точкой" id="float_prec" src="images/fltscale.svg">
```

Эта неточность означает, что часто градиенты, вычисляемые для обновления весов, оказываются равными нулю или бесконечности в глубоких нейронных сетях. Это обычно называют проблемой "исчезающих градиентов" или "взрывающихся градиентов". Это означает, что в алгоритме стохастического градиентного спуска (SGD) веса либо вообще не обновляются, либо стремятся к бесконечности. В любом случае, они не улучшаются в процессе обучения.

Исследователи разработали несколько способов решения этой проблемы, о которых мы поговорим позже в этой книге. Одним из вариантов является изменение определения слоя таким образом, чтобы снизить вероятность возникновения "взрывных" активаций. Мы рассмотрим детали этого процесса в главе, посвященной сверточным нейронным сетям (<<chapter_convolutions>>, когда мы будем говорить о пакетной нормализации), и в главе, посвященной ResNet (<<chapter_resnet>>, когда мы будем говорить о ResNet), хотя эти детали, как правило, не имеют большого значения на практике (если вы не исследователь, разрабатывающий новые подходы к решению этой проблемы). Другая стратегия решения этой проблемы заключается в тщательном выборе начальных значений, что является темой, которую мы рассмотрим в главе, посвященной основам (<<chapter_foundations>>).

Для рекуррентных нейронных сетей (RNN) существуют два типа слоев, которые часто используются для предотвращения "взрывных" активаций: *рекуррентные блоки с управляемым потоком* (GRUs) и *блоки с долгой краткосрочной памятью* (LSTM). Оба этих типа слоев доступны в PyTorch и могут быть использованы вместо обычного слоя RNN. В этой книге мы рассмотрим только LSTM; в интернете можно найти множество хороших руководств, объясняющих GRU, которые являются лишь небольшим вариантом LSTM.

## LSTM (Долгая краткосрочная память)


LSTM – это архитектура, которая была представлена еще в 1997 году Юргеном Шмидхубером и Зеппом Хохрайтером. В этой архитектуре присутствует не одно, а два скрытых состояния. В нашей базовой рекуррентной нейронной сети (RNN), скрытое состояние является результатом работы RNN на предыдущем временном шаге. Это скрытое состояние отвечает за две вещи:

- Предоставление необходимой информации выходному слою для предсказания следующего токена.
- Сохранение памяти обо всем, что произошло в предложении.

Рассмотрим, например, предложения: "Henry has a dog and he likes his dog very much" и "Sophie has a dog and she likes her dog very much". Очевидно, что RNN должна запомнить имя в начале предложения, чтобы иметь возможность предсказать местоимения *he/she* или *his/her*.

На практике, RNN плохо справляются с запоминанием информации, которая была предоставлена гораздо раньше в предложении. Именно поэтому в архитектуре LSTM предусмотрено второе скрытое состояние, называемое *cell state* (состояние ячейки). Состояние ячейки отвечает за сохранение *долговременной памяти*, в то время как основное скрытое состояние фокусируется на предсказании следующего токена. Давайте подробнее рассмотрим, как это реализовано, и построим LSTM с нуля.

### Создание LSTM с нуля.


Для создания модели LSTM необходимо сначала понять ее архитектуру. <<lstm>> демонстрирует ее внутреннюю структуру.

<img src="images/LSTM.png" id="lstm" caption="Архитектура модели LSTM" alt="Схема, показывающая внутреннюю архитектуру модели LSTM" width="700">

На этом рисунке входные данные $x_{t}$ поступают слева вместе с предыдущим скрытым состоянием ($h_{t-1}$) и состоянием ячейки ($c_{t-1}$). Четыре оранжевых блока представляют собой четыре слоя (наши нейронные сети), в которых используется функция активации либо сигмоида ($\sigma$), либо гиперболическая тангенс (tanh). Гиперболическая тангенс — это просто сигмоидальная функция, масштабированная в диапазон от -1 до 1. Ее математическое выражение можно записать следующим образом:

$$\tanh(x) = \frac{e^{x} - e^{-x}}{e^{x}+e^{-x}} = 2 \sigma(2x) - 1$$

где $\sigma$ — сигмоидальная функция. Зеленые круги обозначают операции, выполняемые поэлементно. Справа выводятся новое скрытое состояние ($h_{t}$) и новое состояние ячейки ($c_{t}$), готовые для следующего входного сигнала. Новое скрытое состояние также используется в качестве выходных данных, поэтому стрелка разветвляется, чтобы идти вверх.

Рассмотрим по очереди четыре нейронные сети (называемые *гейтами*) и объясним схему. Но прежде, обратите внимание на то, насколько незначительно меняется состояние ячейки (вверху). Оно даже не проходит напрямую через нейронную сеть! Именно поэтому оно будет поддерживать состояние на более длительный период времени.

Сначала стрелки, представляющие входные данные и предыдущее скрытое состояние, объединяются. В RNN, описанной ранее в этой главе, мы суммировали их. В LSTM они объединяются в один большой тензор. Это означает, что размерность наших эмбеддингов (которая является размерностью $x_{t}$) может отличаться от размерности нашего скрытого состояния. Если мы обозначим эти размеры как `n_in` и `n_hid`, то стрелка внизу имеет размер `n_in + n_hid`; следовательно, все нейронные сети (оранжевые блоки) являются линейными слоями с `n_in + n_hid` входами и `n_hid` выходами.

Первый гейт (если смотреть слева направо) называется *гейтом забывания*. Поскольку это линейный слой, за которым следует сигмоида, его выход будет состоять из скалярных значений в диапазоне от 0 до 1. Мы умножаем этот результат на состояние ячейки, чтобы определить, какую информацию следует сохранить, а какую — отбросить: значения, близкие к 0, отбрасываются, а значения, близкие к 1, сохраняются. Это позволяет LSTM забывать информацию о своем состоянии на длительный период времени. Например, при переходе к новому периоду или токену `xxbos`, мы ожидаем, что она (должна быть обучена) сбросить свое состояние ячейки.

Второй гейт называется *гейтом ввода*. Он работает вместе с третьим гейтом (который не имеет названия, но иногда называется *гейтом ячейки*) для обновления состояния ячейки. Например, мы можем увидеть новое местоимение, и тогда нам нужно заменить информацию о роде, которую гейт забывания удалил. Подобно гейту забывания, гейт ввода решает, какие элементы состояния ячейки следует обновить (значения, близкие к 1), а какие — нет (значения, близкие к 0). Третий гейт определяет, какими будут эти обновленные значения, в диапазоне от -1 до 1 (благодаря функции гиперболической тангенса). Результат затем добавляется к состоянию ячейки.

Последний гейт называется *гейтом вывода*. Он определяет, какую информацию из состояния ячейки следует использовать для генерации выходных данных. Состояние ячейки проходит через функцию гиперболической тангенса, прежде чем быть объединенным с выходным значением сигмоиды от гейта вывода, и результат является новым скрытым состоянием.

С точки зрения кода, мы можем записать те же шаги следующим образом:


In [ ]:
class LSTMCell(Module):
    def __init__(self, ni, nh):
        self.forget_gate = nn.Linear(ni + nh, nh)
        self.input_gate  = nn.Linear(ni + nh, nh)
        self.cell_gate   = nn.Linear(ni + nh, nh)
        self.output_gate = nn.Linear(ni + nh, nh)

    def forward(self, input, state):
        h,c = state
        h = torch.cat([h, input], dim=1)
        forget = torch.sigmoid(self.forget_gate(h))
        c = c * forget
        inp = torch.sigmoid(self.input_gate(h))
        cell = torch.tanh(self.cell_gate(h))
        c = c + inp * cell
        out = torch.sigmoid(self.output_gate(h))
        h = out * torch.tanh(c)
        return h, (h,c)

На практике, мы можем затем рефакторить код. Кроме того, с точки зрения производительности, лучше выполнить одно большое умножение матриц, чем четыре меньших (потому что мы запускаем специальный высокопроизводительный модуль только один раз на GPU, и это позволяет GPU выполнять больше задач параллельно). Операция объединения занимает некоторое время (поскольку нам нужно перемещать один из тензоров на GPU, чтобы все данные находились в одном непрерывном массиве), поэтому мы используем два отдельных слоя для входных данных и для скрытого состояния. Оптимизированный и рефакторизованный код выглядит следующим образом:


In [ ]:
class LSTMCell(Module):
    def __init__(self, ni, nh):
        self.ih = nn.Linear(ni,4*nh)
        self.hh = nn.Linear(nh,4*nh)

    def forward(self, input, state):
        h,c = state
        # One big multiplication for all the gates is better than 4 smaller ones
        gates = (self.ih(input) + self.hh(h)).chunk(4, 1)
        ingate,forgetgate,outgate = map(torch.sigmoid, gates[:3])
        cellgate = gates[3].tanh()

        c = (forgetgate*c) + (ingate*cellgate)
        h = outgate * c.tanh()
        return h, (h,c)

Здесь мы используем метод `chunk` библиотеки PyTorch, чтобы разделить наш тензор на четыре части. Он работает следующим образом:


In [ ]:
t = torch.arange(0,10); t

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [ ]:
t.chunk(2)

(tensor([0, 1, 2, 3, 4]), tensor([5, 6, 7, 8, 9]))

Теперь давайте используем эту архитектуру для обучения языковой модели!

### Обучение языковой модели с использованием рекуррентных нейронных сетей с долгой краткосрочной памятью (LSTM).


Вот та же самая сеть, что и `LMModel5`, использующая двухслойную LSTM-модель. Мы можем обучить ее с использованием более высокой скорости обучения, за более короткое время, и добиться большей точности:

In [ ]:
class LMModel6(Module):
    def __init__(self, vocab_sz, n_hidden, n_layers):
        self.i_h = nn.Embedding(vocab_sz, n_hidden)
        self.rnn = nn.LSTM(n_hidden, n_hidden, n_layers, batch_first=True)
        self.h_o = nn.Linear(n_hidden, vocab_sz)
        self.h = [torch.zeros(n_layers, bs, n_hidden) for _ in range(2)]
        
    def forward(self, x):
        res,h = self.rnn(self.i_h(x), self.h)
        self.h = [h_.detach() for h_ in h]
        return self.h_o(res)
    
    def reset(self): 
        for h in self.h: h.zero_()

In [ ]:
learn = Learner(dls, LMModel6(len(vocab), 64, 2), 
                loss_func=CrossEntropyLossFlat(), 
                metrics=accuracy, cbs=ModelResetter)
learn.fit_one_cycle(15, 1e-2)

epoch,train_loss,valid_loss,accuracy,time
0,3.000821,2.663942,0.438314,00:02
1,2.139642,2.184780,0.240479,00:02
2,1.607275,1.812682,0.439779,00:02
3,1.347711,1.830982,0.497477,00:02
4,1.123113,1.937766,0.594401,00:02
5,0.852042,2.012127,0.631592,00:02
6,0.565494,1.312742,0.725749,00:02
7,0.347445,1.297934,0.711263,00:02
8,0.208191,1.441269,0.731201,00:02
9,0.126335,1.569952,0.737305,00:02


Теперь это лучше, чем многослойная рекуррентная нейронная сеть! Однако, мы все еще видим признаки переобучения, что говорит о том, что небольшая регуляризация может помочь.

## Регуляризация LSTM-сети.

Рекуррентные нейронные сети, как правило, сложно обучать из-за проблемы затухающих активаций и градиентов, о которой мы говорили ранее. Использование ячеек LSTM (или GRU) упрощает обучение по сравнению с обычными рекуррентными сетями, но они все равно подвержены переобучению. Увеличение объема данных, хотя и является возможным подходом, используется реже для текстовых данных, чем для изображений, поскольку в большинстве случаев для создания случайных вариантов требуется другая модель (например, путем перевода текста на другой язык, а затем обратно на исходный язык). В целом, увеличение объема данных для текстовых данных в настоящее время является малоизученной областью.

Однако существуют и другие методы регуляризации, которые можно использовать для снижения переобучения. Эти методы были тщательно изучены для применения с LSTM и описаны в статье ["Regularizing and Optimizing LSTM Language Models"](https://arxiv.org/abs/1708.02182), написанной Стивеном Мерити, Нитишем Шириш Кескаром и Ричардом Сочером. В этой статье показано, как эффективное использование таких методов, как *dropout*, *регуляризация активаций* и *временная регуляризация активаций*, может позволить LSTM превзойти результаты, которые ранее требовали гораздо более сложных моделей. Авторы назвали LSTM, использующий эти методы, *AWD-LSTM*. Мы рассмотрим каждый из этих методов по отдельности.

### Отсев (студентов)


Dropout – это метод регуляризации, который был предложен Джеффри Хинтоном и его коллегами в работе [Improving neural networks by preventing co-adaptation of feature detectors](https://arxiv.org/abs/1207.0580). Основная идея заключается в случайном обнулении некоторых активаций во время обучения. Это обеспечивает активную работу всех нейронов в направлении получения выходного результата, как показано на рисунке <<img_dropout>> (из статьи "Dropout: A Simple Way to Prevent Neural Networks from Overfitting" Нитиша Шривастава и др.).

<img src="images/Dropout1.png" alt="Иллюстрация из статьи, показывающая, как нейроны отключаются при использовании dropout" width="800" id="img_dropout" caption="Применение dropout в нейронной сети (из статьи Нитиша Шривастава и др.)">

Хинтон использовал удачную метафору, объясняя в одном интервью, что послужило источником вдохновения для разработки dropout:

> : Я пришел в свой банк, и кассиры постоянно менялись. Я спросил одного из них, почему, и он ответил, что не знает, но их часто переставляют. Я подумал, что, вероятно, это делается для того, чтобы требовать сотрудничества между сотрудниками для успешного обмана банка. Это заставило меня задуматься о том, что случайное удаление разных подмножеств нейронов для каждого примера может предотвратить сговоры и, следовательно, уменьшить переобучение.

В том же интервью он также объяснил, что дополнительное вдохновение было получено из нейробиологии:

> : Мы не очень хорошо понимаем, почему нейроны генерируют импульсы. Одна из теорий заключается в том, что они стремятся к шуму для регуляризации, потому что у нас гораздо больше параметров, чем точек данных. Идея dropout заключается в том, что, если у вас есть шумные активации, вы можете позволить себе использовать гораздо более сложную модель.

Это объясняет принцип работы метода dropout, который помогает улучшить обобщающую способность модели: во-первых, он способствует более эффективному взаимодействию между нейронами, а во-вторых, он делает активации более "шумными", что делает модель более устойчивой.

Однако, мы можем видеть, что если просто обнулить эти активации, не предпринимая никаких других действий, наша модель столкнется с проблемами при обучении: если мы перейдем от суммы пяти активаций (которые являются положительными числами, поскольку используется функция ReLU) к только двум, это не будет иметь ту же величину. Поэтому, если мы применяем dropout с вероятностью `p`, мы масштабируем все активации, деля их на `1-p` (в среднем `p` будет обнулено, поэтому остается `1-p`), как показано на <<img_dropout1>>.

<img src="images/Dropout.png" alt="Иллюстрация из статьи, представляющая dropout, показывающая, как нейрон включается/выключается" width="600" id="img_dropout1" caption="Почему масштабировать активации при применении dropout (иллюстрация Nitish Srivastava и др.)">

Это полная реализация слоя dropout в PyTorch (хотя нативный слой PyTorch фактически написан на C, а не на Python):


In [ ]:
class Dropout(Module):
    def __init__(self, p): self.p = p
    def forward(self, x):
        if not self.training: return x
        mask = x.new(*x.shape).bernoulli_(1-p)
        return x * mask.div_(1-p)

Метод `bernoulli_` создает тензор, состоящий из случайных нулей (с вероятностью `p`) и единиц (с вероятностью `1-p`), который затем умножается на входные данные и делится на `1-p`. Обратите внимание на использование атрибута `training`, который доступен в любом классе `nn.Module` в PyTorch и указывает, выполняется ли обучение или вывод.

> Обратите внимание: Проводите собственные эксперименты: В предыдущих главах этой книги мы бы добавили пример кода для `bernoulli_`, чтобы вы могли точно увидеть, как он работает. Но теперь, когда вы знаете достаточно, чтобы сделать это самостоятельно, мы будем предоставлять меньше примеров, а вместо этого ожидаем, что вы будете проводить собственные эксперименты, чтобы понять, как все работает. В данном случае, вы увидите в заключительном опросе главы, что мы просим вас поэкспериментировать с `bernoulli_`, но не ждите, пока мы попросим вас экспериментировать, чтобы углубить свое понимание кода, который мы изучаем; просто начните экспериментировать самостоятельно!

Использование dropout перед передачей выходных данных нашего LSTM на финальный слой поможет уменьшить переобучение. Dropout также используется во многих других моделях, включая стандартный CNN, используемый в `fastai.vision`, и доступен в `fastai.tabular` с помощью параметра `ps` (где каждый "p" передается каждому добавленному слою `Dropout`), о чем мы узнаем в разделе <<chapter_arch_details>>.

Метод Dropout работает по-разному в режиме обучения и в режиме валидации, что определяется атрибутом `training` в классе Dropout. Вызов метода `train` для объекта `Module` устанавливает значение `training` в `True` (как для самого модуля, так и для всех модулей, содержащихся в нем рекурсивно), а вызов метода `eval` устанавливает его в `False`. Это происходит автоматически при использовании класса `Learner`, но если вы не используете этот класс, не забудьте переключать этот параметр в зависимости от необходимости.

### Регуляризация активаций и временная регуляризация активаций.


*Регуляризация активаций* (AR) и *временная регуляризация активаций* (TAR) – это два метода регуляризации, очень похожие на метод затухания весов, которые обсуждаются в <<chapter_collab>>. При использовании метода затухания весов мы добавляем небольшое штрафное значение к функции потерь, которое направлено на минимизацию весов. В случае регуляризации активаций, мы пытаемся минимизировать значения конечных активаций, генерируемых LSTM, а не сами веса.

Чтобы регуляризовать конечные активации, необходимо сохранить их где-то, а затем добавить среднее значение квадратов этих активаций к функции потерь (вместе с коэффициентом `alpha`, который аналогичен коэффициенту `wd` для метода затухания весов):

```python
loss += alpha * activations.pow(2).mean()
```

Регуляризация временной активации связана с тем, что мы предсказываем токены в предложении. Это означает, что, вероятно, выходные данные наших LSTM должны иметь определенный смысл, если мы читаем их последовательно. TAR (Temporal Activation Regularization) предназначена для стимулирования такого поведения, добавляя штраф к функции потерь, чтобы минимизировать разницу между двумя последовательными активациями. Тензор наших активаций имеет форму `bs x sl x n_hid`, и мы читаем последовательные активации по оси длины последовательности (посреднему измерению). Таким образом, TAR можно выразить следующим образом:

```python
loss += beta * (activations[:,1:] - activations[:,:-1]).pow(2).mean()
```

`alpha` и `beta` — это два гиперпараметра, которые необходимо настроить. Чтобы это работало, наша модель с dropout должна возвращать три вещи: правильный выход, активации LSTM до применения dropout и активации LSTM после применения dropout. AR (Activation Regularization) часто применяется к активациям, к которым был применен dropout (чтобы не штрафовать активации, которые впоследствии были преобразованы в нули), в то время как TAR применяется к активациям, к которым dropout не применялся (потому что эти нули создают большие различия между двумя последовательными временными шагами). Существует также функция обратного вызова под названием `RNNRegularizer`, которая будет применять эту регуляризацию для нас.

### Обучение регуляризованной LSTM с фиксированными весами.

Мы можем объединить метод dropout (который применяется перед тем, как мы переходим к выходному слою) с методами AR и TAR для обучения нашей предыдущей модели LSTM. Нам просто нужно возвращать три значения вместо одного: обычный выход нашей LSTM, значения, полученные после применения dropout, и значения из наших LSTM. Последние два будут использоваться колбэком `RNNRegularization` для внесения вклада в функцию потерь.

Еще один полезный прием, который мы можем позаимствовать из [статьи про LSTM AWD](https://arxiv.org/abs/1708.02182), – это *связывание весов*. В языковой модели входные эмбеддинги представляют собой отображение от английских слов к активациям, а выходной скрытый слой представляет собой отображение от активаций к английским словам. Интуитивно, мы можем предположить, что эти отображения могут быть одинаковыми. Мы можем реализовать это в PyTorch, присвоив одинаковую матрицу весов каждому из этих слоев:

    self.h_o.weight = self.i_h.weight

В модели `LMModel7` мы включаем эти финальные улучшения:


In [ ]:
class LMModel7(Module):
    def __init__(self, vocab_sz, n_hidden, n_layers, p):
        self.i_h = nn.Embedding(vocab_sz, n_hidden)
        self.rnn = nn.LSTM(n_hidden, n_hidden, n_layers, batch_first=True)
        self.drop = nn.Dropout(p)
        self.h_o = nn.Linear(n_hidden, vocab_sz)
        self.h_o.weight = self.i_h.weight
        self.h = [torch.zeros(n_layers, bs, n_hidden) for _ in range(2)]
        
    def forward(self, x):
        raw,h = self.rnn(self.i_h(x), self.h)
        out = self.drop(raw)
        self.h = [h_.detach() for h_ in h]
        return self.h_o(out),raw,out
    
    def reset(self): 
        for h in self.h: h.zero_()

Мы можем создать объект `Learner` с применением регуляризации, используя обратный вызов `RNNRegularizer`:


In [ ]:
learn = Learner(dls, LMModel7(len(vocab), 64, 2, 0.5),
                loss_func=CrossEntropyLossFlat(), metrics=accuracy,
                cbs=[ModelResetter, RNNRegularizer(alpha=2, beta=1)])

Класс `TextLearner` автоматически добавляет эти два обработчика событий (с указанными значениями параметров `alpha` и `beta` по умолчанию), поэтому мы можем упростить предыдущую строку следующим образом:


In [ ]:
learn = TextLearner(dls, LMModel7(len(vocab), 64, 2, 0.4),
                    loss_func=CrossEntropyLossFlat(), metrics=accuracy)

Затем мы можем обучить модель и добавить дополнительную регуляризацию, увеличив коэффициент затухания весов до `0.1`:

In [ ]:
learn.fit_one_cycle(15, 1e-2, wd=0.1)

epoch,train_loss,valid_loss,accuracy,time
0,2.693885,2.013484,0.466634,00:02
1,1.685549,1.187310,0.629313,00:02
2,0.973307,0.791398,0.745605,00:02
3,0.555823,0.640412,0.794108,00:02
4,0.351802,0.557247,0.836100,00:02
5,0.244986,0.594977,0.807292,00:02
6,0.192231,0.511690,0.846761,00:02
7,0.162456,0.520370,0.858073,00:02
8,0.142664,0.525918,0.842285,00:02
9,0.128493,0.495029,0.858073,00:02


Вот это уже намного лучше, чем наша предыдущая модель!

## Заключение


Теперь вы увидели все, что находится внутри архитектуры AWD-LSTM, которую мы использовали для классификации текста в главе <<chapter_nlp>>. В ней используется механизм dropout во многих местах:

- Dropout для эмбеддингов (внутри слоя эмбеддингов, отбрасываются некоторые случайные векторы эмбеддингов)
- Dropout для входных данных (применяется после слоя эмбеддингов)
- Dropout для весов (применяется к весам LSTM на каждом шаге обучения)
- Dropout для скрытых состояний (применяется к скрытому состоянию между двумя слоями)

Это делает ее еще более устойчивой к переобучению. Поскольку тонкая настройка этих пяти параметров dropout (включая dropout перед выходным слоем) является сложной задачей, мы определили хорошие значения по умолчанию и позволяем регулировать общую величину dropout с помощью параметра `drop_mult`, который вы видели в той главе (и который умножается на каждый параметр dropout).

Еще одна очень мощная архитектура, особенно эффективная в задачах "sequence-to-sequence" (то есть в задачах, где зависимая переменная сама является последовательностью переменной длины, например, машинный перевод), – это архитектура Transformers. Вы можете найти информацию о ней в дополнительной главе на [сайте книги](https://book.fast.ai/).

## Анкета


1. Если набор данных для вашего проекта настолько велик и сложен, что работа с ним занимает значительное время, что вы должны сделать?
2. Почему мы объединяем документы в нашем наборе данных перед созданием языковой модели?
3. Чтобы использовать стандартную полностью связанную сеть для предсказания четвертого слова, исходя из трех предыдущих слов, какие два изменения необходимо внести в нашу модель?
4. Как можно использовать одну и ту же матрицу весов в нескольких слоях в PyTorch?
5. Напишите модуль, который предсказывает третье слово, исходя из двух предыдущих слов предложения, не "подглядывая".
6. Что такое рекуррентная нейронная сеть?
7. Что такое "скрытое состояние"?
8. Что в модели `LMModel1` соответствует "скрытому состоянию"?
9. Почему важно передавать текст в модель последовательно, чтобы поддерживать состояние в RNN?
10. Что такое "развернутое" представление RNN?
11. Почему поддержание скрытого состояния в RNN может приводить к проблемам с памятью и производительностью? Как решить эту проблему?
12. Что такое "BPTT"?
13. Напишите код для вывода нескольких первых пакетов из набора данных для проверки, включая преобразование идентификаторов токенов обратно в английские строки, как мы показывали для пакетов данных IMDb в <<chapter_nlp>>.
14. Что делает обратный вызов `ModelResetter`? Зачем он нужен?
15. Какие недостатки есть в предсказании только одного выходного слова для каждой группы из трех входных слов?
16. Почему нам нужна собственная функция потерь для `LMModel4`?
17. Почему обучение `LMModel4` нестабильно?
18. В развернутом представлении мы видим, что рекуррентная нейронная сеть на самом деле имеет много слоев. Так зачем же мы объединяем RNN, чтобы получить лучшие результаты?
19. Нарисуйте схему многослойной (стеклянной) RNN.
20. Почему мы должны получать лучшие результаты в RNN, если мы реже вызываем `detach`? Почему это может не происходить на практике с простой RNN?
21. Почему глубокая сеть может приводить к очень большим или очень маленьким активациям? Почему это важно?
22. В представлении чисел с плавающей точкой, используемом компьютером, какие числа наиболее точны?
23. Почему "исчезающие градиенты" препятствуют обучению?
24. Почему наличие двух скрытых состояний в архитектуре LSTM помогает? Какова цель каждого из них?
25. Как называются эти два состояния в LSTM?
26. Что такое tanh, и как он связан с sigmoid?
27. Какова цель этого кода в `LSTMCell`: `h = torch.cat([h, input], dim=1)`?
28. Что делает функция `chunk` в PyTorch?
29. Внимательно изучите рефакторизованную версию `LSTMCell`, чтобы убедиться, что вы понимаете, как и почему она делает то же самое, что и нерефакторизованная версия.
30. Почему мы можем использовать более высокую скорость обучения для `LMModel6`?
31. Какие три метода регуляризации используются в модели AWD-LSTM?
32. Что такое "dropout"?
33. Почему мы масштабируем активации с помощью dropout? Применяется ли это во время обучения, во время предсказания или в обоих случаях?
34. Какова цель этой строки из `Dropout`: `if not self.training: return x`?
35. Поэкспериментируйте с `bernoulli_`, чтобы понять, как он работает.
36. Как установить модель в режим обучения в PyTorch? В режим оценки?
37. Напишите уравнение для регуляризации активаций (в виде математической формулы или кода, как вам удобнее). Чем она отличается от регуляризации весов?
38. Напишите уравнение для временной регуляризации активаций (в виде математической формулы или кода, как вам удобнее). Почему мы бы не использовали это для задач компьютерного зрения?
39. Что такое "привязка весов" в языковой модели?


### Дальнейшие исследования


1. В классе `LMModel2`, почему функция `forward` может начинаться с `h=0`? Зачем нам не нужно писать `h=torch.zeros(...)`?
2. Напишите код для LSTM с нуля (можно использовать ссылку <<lstm>>).
3. Найдите в интернете информацию об архитектуре GRU, реализуйте ее с нуля и попробуйте обучить модель. Попробуйте добиться результатов, аналогичных тем, которые мы видели в этой главе. Сравните ваши результаты с результатами встроенного модуля `GRU` в PyTorch.
4. Изучите исходный код AWD-LSTM в библиотеке fastai и попытайтесь сопоставить каждую строку кода с концепциями, представленными в этой главе.